In [ ]:
# 🤖 AutoGen Multi-Agent System with Kafka Pub/Sub
# Smart Home Monitoring - AutoGen agents communicating via Kafka broker

import os, sys, asyncio, random, json
from datetime import datetime
from typing import List

# --------- Dependency Installation ---------
def ensure_package(package_name: str):
    module_name = package_name.split("[")[0].replace("-", "_")
    try:
        __import__(module_name)
        return
    except ImportError:
        pass
    import subprocess
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package_name],
        stdout=subprocess.DEVNULL
    )

required_packages = ["autogen-core", "autogen-ext", "kafka-python", "anthropic>=0.34"]
for pkg in required_packages:
    ensure_package(pkg)

# --------- Core Imports ---------
from dataclasses import dataclass, asdict
from kafka import KafkaProducer, KafkaConsumer
from autogen_core import (
    AgentId, 
    TypeSubscription,
    DefaultTopicId,
    MessageContext,
    RoutedAgent,
    default_subscription,
    message_handler,
    SingleThreadedAgentRuntime
)
import anthropic

# --------- Configuration ---------
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not API_KEY:
    raise ValueError("❌ ANTHROPIC_API_KEY environment variable is required")

MODEL_ID = "claude-sonnet-4-5-20250929"
TOKEN_LIMIT = 250
KAFKA_BOOTSTRAP_SERVERS = ['localhost:9092']
TOTAL_CYCLES = 8  # Number of sensing cycles

# Kafka Topics (Pub/Sub channels)
TOPIC_TEMPERATURE = "home.temperature"
TOPIC_MOTION = "home.motion"
TOPIC_ALERTS = "home.alerts"

# --------- Message Protocol (AutoGen Messages) ---------
@dataclass
class SensorReading:
    """AutoGen message published by sensor agents"""
    sensor_type: str  # "temperature" or "motion"
    sensor_id: str
    location: str
    data: dict
    timestamp: str
    cycle: int
    
    def to_json(self) -> str:
        return json.dumps(asdict(self))
    
    @classmethod
    def from_json(cls, json_str: str):
        return cls(**json.loads(json_str))

@dataclass
class Alert:
    """AutoGen message published by alert manager"""
    alert_type: str
    severity: str
    message: str
    source_reading: dict
    timestamp: str
    
    def to_json(self) -> str:
        return json.dumps(asdict(self))

@dataclass
class StartPublishing:
    """Control message to start publishing"""
    cycles: int

# --------- AUTOGEN AGENT 1: Temperature Sensor ---------
@default_subscription
class TemperatureSensorAgent(RoutedAgent):
    """
    AutoGen RoutedAgent that publishes temperature readings to Kafka
    Demonstrates PUBLISHER in pub/sub architecture using AutoGen framework
    """
    
    def __init__(self, sensor_id: str, location: str) -> None:
        super().__init__("Temperature sensor agent")
        self.sensor_id = sensor_id
        self.location = location
        self.cycle_count = 0
        
        # Initialize Kafka Producer
        self.producer = KafkaProducer(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            value_serializer=lambda v: v.encode('utf-8')
        )
        self.topic = TOPIC_TEMPERATURE
        print(f"✅ [AutoGen] TemperatureSensorAgent initialized: {sensor_id} at {location}")
    
    def generate_reading(self) -> SensorReading:
        """Generate simulated temperature data"""
        temp = round(random.uniform(18.0, 28.0), 1)
        return SensorReading(
            sensor_type="temperature",
            sensor_id=self.sensor_id,
            location=self.location,
            data={"temperature_celsius": temp},
            timestamp=datetime.now().isoformat(),
            cycle=self.cycle_count
        )
    
    @message_handler
    async def handle_start_publishing(self, message: StartPublishing, ctx: MessageContext) -> None:
        """AutoGen message handler - starts publishing when triggered"""
        print(f"🌡️  [AutoGen Agent] {self.sensor_id} received StartPublishing message")
        
        for _ in range(message.cycles):
            self.cycle_count += 1
            reading = self.generate_reading()
            
            print(f"🌡️  [Temp-{self.sensor_id}] Publishing to Kafka: {reading.data['temperature_celsius']}°C at {self.location}")
            
            # Publish to Kafka topic
            self.producer.send(self.topic, value=reading.to_json())
            self.producer.flush()
            
            await asyncio.sleep(2)
        
        print(f"✓ [AutoGen] TemperatureSensorAgent completed publishing")
        self.producer.close()

# --------- AUTOGEN AGENT 2: Motion Sensor ---------
@default_subscription
class MotionSensorAgent(RoutedAgent):
    """
    AutoGen RoutedAgent that publishes motion events to Kafka
    Demonstrates PUBLISHER in pub/sub architecture using AutoGen framework
    """
    
    def __init__(self, sensor_id: str, location: str) -> None:
        super().__init__("Motion sensor agent")
        self.sensor_id = sensor_id
        self.location = location
        self.cycle_count = 0
        
        # Initialize Kafka Producer
        self.producer = KafkaProducer(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            value_serializer=lambda v: v.encode('utf-8')
        )
        self.topic = TOPIC_MOTION
        print(f"✅ [AutoGen] MotionSensorAgent initialized: {sensor_id} at {location}")
    
    def generate_reading(self) -> SensorReading:
        """Generate simulated motion data"""
        motion_detected = random.choice([True, False, False, False])
        return SensorReading(
            sensor_type="motion",
            sensor_id=self.sensor_id,
            location=self.location,
            data={
                "motion_detected": motion_detected,
                "confidence": round(random.uniform(0.75, 0.99), 2)
            },
            timestamp=datetime.now().isoformat(),
            cycle=self.cycle_count
        )
    
    @message_handler
    async def handle_start_publishing(self, message: StartPublishing, ctx: MessageContext) -> None:
        """AutoGen message handler - starts publishing when triggered"""
        print(f"👁️  [AutoGen Agent] {self.sensor_id} received StartPublishing message")
        
        for _ in range(message.cycles):
            self.cycle_count += 1
            reading = self.generate_reading()
            
            status = "🚶 MOTION" if reading.data['motion_detected'] else "🔇 Clear"
            print(f"{status} [Motion-{self.sensor_id}] Publishing to Kafka: {self.location}")
            
            # Publish to Kafka topic
            self.producer.send(self.topic, value=reading.to_json())
            self.producer.flush()
            
            await asyncio.sleep(3)
        
        print(f"✓ [AutoGen] MotionSensorAgent completed publishing")
        self.producer.close()

# --------- AUTOGEN AGENT 3: Alert Manager with Claude AI ---------
@default_subscription
class AlertManagerAgent(RoutedAgent):
    """
    AutoGen RoutedAgent that SUBSCRIBES to Kafka topics and uses Claude AI
    Demonstrates SUBSCRIBER in pub/sub architecture using AutoGen framework
    """
    
    def __init__(self, completion_signal: asyncio.Event) -> None:
        super().__init__("Alert manager agent")
        self.completion_signal = completion_signal
        self.llm_client = anthropic.Anthropic(api_key=API_KEY)
        
        # Temperature thresholds
        self.temp_threshold_high = 26.0
        self.temp_threshold_low = 19.0
        
        # Statistics
        self.readings_received = 0
        self.alerts_generated = 0
        
        # Initialize Kafka Consumer with unique group ID for each run
        import uuid
        run_id = str(uuid.uuid4())[:8]
        
        self.consumer = KafkaConsumer(
            TOPIC_TEMPERATURE,
            TOPIC_MOTION,
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            value_deserializer=lambda m: m.decode('utf-8'),
            auto_offset_reset='earliest',  # Read from beginning to catch all messages
            group_id=f'alert-manager-{run_id}',  # Unique group per run
            enable_auto_commit=False,  # Don't commit offsets
            consumer_timeout_ms=5000  # Timeout if no messages for 5 seconds
        )
        
        # Initialize Kafka Producer for alerts
        self.producer = KafkaProducer(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            value_serializer=lambda v: v.encode('utf-8')
        )
        self.alert_topic = TOPIC_ALERTS
        print(f"✅ [AutoGen] AlertManagerAgent initialized with Claude AI")
    
    def _query_claude(self, reading: SensorReading, alert_context: str) -> str:
        """Use Claude AI to generate contextual alert"""
        try:
            prompt = f"""You are a smart home AI. Generate a brief alert (max 2 sentences).

Context: {alert_context}
Sensor: {reading.sensor_type} at {reading.location}
Data: {json.dumps(reading.data)}

Provide a clear alert."""

            response = self.llm_client.messages.create(
                model=MODEL_ID,
                max_tokens=TOKEN_LIMIT,
                messages=[{"role": "user", "content": prompt}]
            )
            
            text_blocks = []
            for block in response.content:
                if hasattr(block, "text"):
                    text_blocks.append(block.text)
            
            return " ".join(text_blocks).strip()
            
        except Exception as e:
            return f"Alert: {alert_context}"
    
    def process_temperature(self, reading: SensorReading) -> None:
        """Process temperature readings"""
        temp = reading.data['temperature_celsius']
        
        if temp > self.temp_threshold_high:
            alert_context = f"Temperature HIGH at {temp}°C"
            ai_message = self._query_claude(reading, alert_context)
            
            alert = Alert(
                alert_type="HIGH_TEMPERATURE",
                severity="WARNING",
                message=ai_message,
                source_reading=reading.data,
                timestamp=datetime.now().isoformat()
            )
            
            print(f"\n🔥 [AutoGen AlertManager] HIGH TEMP ALERT")
            print(f"   Location: {reading.location}")
            print(f"   Claude AI: {ai_message}\n")
            
            self.producer.send(self.alert_topic, value=alert.to_json())
            self.producer.flush()
            self.alerts_generated += 1
            
        elif temp < self.temp_threshold_low:
            alert_context = f"Temperature LOW at {temp}°C"
            ai_message = self._query_claude(reading, alert_context)
            
            alert = Alert(
                alert_type="LOW_TEMPERATURE",
                severity="INFO",
                message=ai_message,
                source_reading=reading.data,
                timestamp=datetime.now().isoformat()
            )
            
            print(f"\n❄️  [AutoGen AlertManager] LOW TEMP ALERT")
            print(f"   Location: {reading.location}")
            print(f"   Claude AI: {ai_message}\n")
            
            self.producer.send(self.alert_topic, value=alert.to_json())
            self.producer.flush()
            self.alerts_generated += 1
    
    def process_motion(self, reading: SensorReading) -> None:
        """Process motion readings"""
        if reading.data['motion_detected']:
            confidence = reading.data['confidence']
            alert_context = f"Motion detected ({confidence*100}% confidence)"
            ai_message = self._query_claude(reading, alert_context)
            
            alert = Alert(
                alert_type="MOTION_DETECTED",
                severity="INFO",
                message=ai_message,
                source_reading=reading.data,
                timestamp=datetime.now().isoformat()
            )
            
            print(f"\n👁️  [AutoGen AlertManager] MOTION ALERT")
            print(f"   Location: {reading.location}")
            print(f"   Claude AI: {ai_message}\n")
            
            self.producer.send(self.alert_topic, value=alert.to_json())
            self.producer.flush()
            self.alerts_generated += 1
    
    @message_handler
    async def handle_start_subscribing(self, message: StartPublishing, ctx: MessageContext) -> None:
        """AutoGen message handler - starts Kafka subscription"""
        print("🤖 [AutoGen AlertManager] Starting Kafka subscription...\n")
        
        max_cycles_seen = 0
        cycles_target = message.cycles
        no_message_count = 0
        max_no_message = 15  # Wait longer - 15 attempts with no messages
        
        while max_cycles_seen < cycles_target or no_message_count < max_no_message:
            # Poll Kafka for messages
            messages = self.consumer.poll(timeout_ms=1000)
            
            if not messages:
                no_message_count += 1
                # Only stop if we've seen all cycles AND no new messages
                if max_cycles_seen >= cycles_target and no_message_count >= max_no_message:
                    break
                await asyncio.sleep(0.5)
                continue
            else:
                no_message_count = 0  # Reset counter when we get messages
            
            for topic_partition, records in messages.items():
                topic = topic_partition.topic
                
                for record in records:
                    self.readings_received += 1
                    
                    try:
                        reading = SensorReading.from_json(record.value)
                        max_cycles_seen = max(max_cycles_seen, reading.cycle)
                        
                        if topic == TOPIC_TEMPERATURE:
                            self.process_temperature(reading)
                        elif topic == TOPIC_MOTION:
                            self.process_motion(reading)
                            
                    except Exception as e:
                        print(f"⚠️  Error: {e}")
            
            await asyncio.sleep(0.1)
        
        print(f"\n{'='*70}")
        print(f"📊 [AutoGen AlertManager] Final Statistics:")
        print(f"   Readings received: {self.readings_received}")
        print(f"   Alerts generated: {self.alerts_generated}")
        print(f"   Max cycle seen: {max_cycles_seen}/{cycles_target}")
        print(f"{'='*70}\n")
        
        self.consumer.close()
        self.producer.close()
        self.completion_signal.set()

# --------- Main Orchestration with AutoGen Runtime ---------
async def run_autogen_kafka_system():
    """
    Demonstrates Pub/Sub using:
    - AutoGen Core framework (RoutedAgent base class, decorators)
    - Kafka as message broker
    - Claude AI for intelligence
    """
    
    completion_flag = asyncio.Event()
    
    print("=" * 70)
    print("🏠 AUTOGEN + KAFKA PUB/SUB SMART HOME SYSTEM")
    print("=" * 70)
    print("📋 Architecture:")
    print("   - AutoGen Core: Agent framework (RoutedAgent, @decorators)")
    print("   - Kafka: Message broker for pub/sub")
    print("   - Claude AI: Intelligent processing")
    print("=" * 70 + "\n")
    
    # Create AutoGen runtime
    runtime = SingleThreadedAgentRuntime()
    
    # Register AutoGen agents
    print("📦 Registering AutoGen agents...\n")
    
    # Create agent instances (AutoGen RoutedAgents)
    temp_agent = TemperatureSensorAgent("T001", "Living Room")
    motion_agent = MotionSensorAgent("M001", "Front Door")
    alert_agent = AlertManagerAgent(completion_flag)
    
    # Register with AutoGen runtime
    await TemperatureSensorAgent.register(
        runtime,
        "temperature_sensor",
        lambda: temp_agent
    )
    
    await MotionSensorAgent.register(
        runtime,
        "motion_sensor",
        lambda: motion_agent
    )
    
    await AlertManagerAgent.register(
        runtime,
        "alert_manager",
        lambda: alert_agent
    )
    
    print("\n" + "=" * 70)
    print("🎬 STARTING AUTOGEN AGENTS CONCURRENTLY")
    print("=" * 70 + "\n")
    
    # Start runtime
    runtime.start()
    
    # Run agents concurrently by calling their handlers directly
    # This maintains AutoGen structure while allowing concurrent execution
    start_msg = StartPublishing(cycles=TOTAL_CYCLES)
    
    # Create concurrent tasks for all agents
    tasks = [
        asyncio.create_task(alert_agent.handle_start_subscribing(start_msg, None)),
        asyncio.create_task(temp_agent.handle_start_publishing(start_msg, None)),
        asyncio.create_task(motion_agent.handle_start_publishing(start_msg, None))
    ]
    
    # Wait for completion signal or timeout
    try:
        await asyncio.wait_for(completion_flag.wait(), timeout=60)
    except asyncio.TimeoutError:
        print("\n⏱️  Timeout - forcing completion")
    
    # Ensure all tasks complete
    await asyncio.gather(*tasks, return_exceptions=True)
    
    # Stop runtime
    await runtime.stop()
    
    print("=" * 70)
    print("✅ AutoGen agents terminated successfully")
    print("=" * 70 + "\n")

# --------- Execute in Jupyter ---------
await run_autogen_kafka_system()

🏠 AUTOGEN + KAFKA PUB/SUB SMART HOME SYSTEM
📋 Architecture:
   - AutoGen Core: Agent framework (RoutedAgent, @decorators)
   - Kafka: Message broker for pub/sub
   - Claude AI: Intelligent processing

📦 Registering AutoGen agents...

✅ [AutoGen] TemperatureSensorAgent initialized: T001 at Living Room
✅ [AutoGen] MotionSensorAgent initialized: M001 at Front Door
✅ [AutoGen] AlertManagerAgent initialized with Claude AI

🎬 STARTING AUTOGEN AGENTS CONCURRENTLY

🤖 [AutoGen AlertManager] Starting Kafka subscription...

🌡️  [AutoGen Agent] T001 received StartPublishing message
🌡️  [Temp-T001] Publishing to Kafka: 18.8°C at Living Room
👁️  [AutoGen Agent] M001 received StartPublishing message
🔇 Clear [Motion-M001] Publishing to Kafka: Front Door

🔥 [AutoGen AlertManager] HIGH TEMP ALERT
   Location: Living Room
   Claude AI: **Temperature Alert:** Living Room temperature is high at 26.6°C. Consider adjusting your thermostat or opening windows to cool down the space.


🔥 [AutoGen AlertManager] 